# 22 — Alpha Biological Analysis

Deep analysis of per-gene integration coefficients (alpha) and delta-alpha.
Covers: alpha landscape, top genes, biological interpretation, cross-seed reproducibility, S_g/FQR analysis.

**Key biological question**: Does alpha capture meaningful evolutionary conservation beyond sequence identity?
- Delta-alpha > 0: gene more conserved in expression than sequence suggests
- Delta-alpha < 0: gene more divergent in expression than sequence suggests

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="scvi")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import anndata as ad
import torch
from pathlib import Path
from scipy.stats import spearmanr, mannwhitneyu

from chimera._model import ChimeraModel

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
REPO = Path("..").resolve()

In [ ]:
# === Configuration ===
ADATA_PATH = REPO / "data" / "processed" / "camex_human_monkey.h5ad"
RUN_DIR = REPO / "training" / "runs" / "iter28-baseline"
ORTHOLOG_PATH = REPO / "data" / "processed" / "ortholog_table.csv"

adata = ad.read_h5ad(ADATA_PATH)
ChimeraModel.setup_anndata(adata, species_key="species", batch_key="batch")
model = ChimeraModel.load(str(RUN_DIR / "model"), adata=adata)

# Extract outputs
alpha = model.get_alpha()
delta_alpha = model.get_delta_alpha()
correction = model.get_correction()
S_g = model.get_S_g()
prior = model.module.conservation_prior.cpu().numpy()
gene_names = adata.var_names.values

# Gene metadata
gene_df = pd.DataFrame({
    "gene": gene_names,
    "alpha": alpha,
    "delta_alpha": delta_alpha,
    "correction": correction,
    "prior": prior,
    "S_g": S_g,
    "abs_correction": np.abs(correction),
}).set_index("gene")

# Load ortholog table if available
if ORTHOLOG_PATH.exists():
    ortho = pd.read_csv(ORTHOLOG_PATH)
    print(f"Ortholog table: {ortho.shape}")
else:
    ortho = None
    print("No ortholog table found")

print(f"Genes: {len(gene_df)}")
print(f"Alpha: [{alpha.min():.3f}, {alpha.max():.3f}], mean={alpha.mean():.3f}")
print(f"Delta-alpha: [{delta_alpha.min():.3f}, {delta_alpha.max():.3f}]")

## 1. Alpha Landscape

In [ ]:
fig = plt.figure(figsize=(16, 8))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# 1a. Alpha distribution with prior overlay
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(alpha, bins=60, range=(0, 1), alpha=0.7, color="steelblue", edgecolor="none", density=True, label="Learned alpha")
ax1.hist(prior, bins=60, range=(0, 1), alpha=0.3, color="gray", edgecolor="none", density=True, label="Prior")
ax1.set_xlabel("Alpha"); ax1.set_ylabel("Density")
ax1.set_title("Alpha Distribution")
ax1.legend(fontsize=8)

# 1b. Delta-alpha distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(delta_alpha, bins=80, color="coral", edgecolor="none", alpha=0.8)
ax2.axvline(0, color="black", linestyle="--", linewidth=1)
ax2.axvline(0.02, color="green", linestyle=":", alpha=0.5)
ax2.axvline(-0.02, color="green", linestyle=":", alpha=0.5)
ax2.set_xlabel("Delta-alpha")
ax2.set_title(f"Delta-Alpha (pos>0.02: {(delta_alpha>0.02).mean():.1%}, neg<-0.02: {(delta_alpha<-0.02).mean():.1%})")

# 1c. Alpha vs prior scatter with density
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(prior, alpha, s=1, alpha=0.2, c="steelblue", rasterized=True)
ax3.plot([0, 1], [0, 1], "r--", linewidth=1, alpha=0.7, label="alpha = prior")
r = spearmanr(prior, alpha).correlation
ax3.set_xlabel("Conservation prior"); ax3.set_ylabel("Learned alpha")
ax3.set_title(f"Alpha vs Prior (r={r:.3f})")
ax3.legend(fontsize=8)

# 1d. Delta-alpha vs prior (key biological plot)
ax4 = fig.add_subplot(gs[1, 0])
ax4.scatter(prior, delta_alpha, s=1, alpha=0.2, c="coral", rasterized=True)
ax4.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax4.set_xlabel("Conservation prior"); ax4.set_ylabel("Delta-alpha")
ax4.set_title("Delta-Alpha vs Prior")

# 1e. Correction distribution
ax5 = fig.add_subplot(gs[1, 1])
ax5.hist(correction, bins=80, color="purple", alpha=0.7, edgecolor="none")
ax5.axvline(0, color="black", linestyle="--", linewidth=0.8)
ax5.set_xlabel("GIN correction (logit space)")
ax5.set_title(f"Correction Distribution (mean|c|={np.abs(correction).mean():.3f})")

# 1f. Alpha quintile summary
ax6 = fig.add_subplot(gs[1, 2])
quintiles = pd.qcut(alpha, 5, labels=["Q1\n(low)", "Q2", "Q3", "Q4", "Q5\n(high)"])
q_stats = gene_df.groupby(quintiles, observed=True).agg(
    alpha_mean=("alpha", "mean"),
    delta_mean=("delta_alpha", "mean"),
    prior_mean=("prior", "mean"),
    n_genes=("alpha", "count"),
)
x = range(5)
ax6.bar(x, q_stats["alpha_mean"], color="steelblue", alpha=0.7, label="alpha mean")
ax6.bar(x, q_stats["prior_mean"], color="gray", alpha=0.3, label="prior mean")
ax6.set_xticks(x); ax6.set_xticklabels(q_stats.index)
ax6.set_ylabel("Mean value"); ax6.set_title("Alpha Quintiles vs Prior")
ax6.legend(fontsize=7)

fig.suptitle("Alpha Landscape", fontsize=14)
plt.show()

## 2. Top Delta-Alpha Genes

Genes where the model learned the most — expression conservation diverges from sequence conservation.

In [ ]:
N_TOP = 20

# Top positive delta-alpha (more conserved in expression than sequence)
top_pos = gene_df.nlargest(N_TOP, "delta_alpha")[["alpha", "prior", "delta_alpha", "S_g", "correction"]]
print(f"=== Top {N_TOP} Positive Delta-Alpha (expression MORE conserved than sequence) ===")
print("These genes have conserved cross-species expression despite lower sequence identity.\n")
display(top_pos.style.format({"alpha": "{:.3f}", "prior": "{:.3f}", "delta_alpha": "{:.3f}",
                              "S_g": "{:.3f}", "correction": "{:.3f}"}))

print(f"\n=== Top {N_TOP} Negative Delta-Alpha (expression LESS conserved than sequence) ===")
print("These genes have divergent cross-species expression despite high sequence identity.\n")
top_neg = gene_df.nsmallest(N_TOP, "delta_alpha")[["alpha", "prior", "delta_alpha", "S_g", "correction"]]
display(top_neg.style.format({"alpha": "{:.3f}", "prior": "{:.3f}", "delta_alpha": "{:.3f}",
                              "S_g": "{:.3f}", "correction": "{:.3f}"}))

## 3. Expression Patterns of Top Genes

Show per-species expression of top delta-alpha genes to visually confirm the alpha assignments make biological sense.

In [ ]:
import scipy.sparse as sp_mod

X = adata.X.toarray() if sp_mod.issparse(adata.X) else np.array(adata.X)
lib = X.sum(axis=1, keepdims=True).clip(min=1)
cpm = X / lib * 1e4  # CPM normalization

species_arr = adata.obs["species"].values
unique_sp = sorted(adata.obs["species"].unique())

def plot_top_genes(gene_list, title, n_show=10):
    genes = gene_list[:n_show]
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for ax, g in zip(axes.flat, genes):
        g_idx = np.where(gene_names == g)[0][0]
        for sp in unique_sp:
            sp_mask = species_arr == sp
            vals = np.log1p(cpm[sp_mask, g_idx])
            ax.hist(vals, bins=30, alpha=0.5, label=sp, density=True, edgecolor="none")
        ax.set_title(f"{g}\nalpha={alpha[g_idx]:.2f}, prior={prior[g_idx]:.2f}", fontsize=8)
        ax.tick_params(labelsize=7)
    axes[0, 0].legend(fontsize=7)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()

plot_top_genes(top_pos.index.tolist(), "Top Positive Delta-Alpha Genes (conserved expression, lower seq identity)")
plot_top_genes(top_neg.index.tolist(), "Top Negative Delta-Alpha Genes (divergent expression, higher seq identity)")

## 4. Alpha vs Expression Divergence

Direct validation: does alpha correlate with actual cross-species expression similarity?

In [ ]:
# Compute per-gene expression divergence between species
# Use library-size-normalized mean per species
sp_profiles = {}
for sp in unique_sp:
    mask = species_arr == sp
    sp_profiles[sp] = np.log1p(1e4 * (X[mask] / lib[mask]).mean(axis=0))

# Expression divergence = |log(mean_A) - log(mean_B)|
expr_div = np.abs(sp_profiles[unique_sp[0]] - sp_profiles[unique_sp[1]])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Alpha vs expression divergence
axes[0].scatter(alpha, expr_div, s=1, alpha=0.2, c="steelblue", rasterized=True)
r1 = spearmanr(alpha, expr_div).correlation
axes[0].set_xlabel("Alpha"); axes[0].set_ylabel("|log expr divergence|")
axes[0].set_title(f"Alpha vs Expression Divergence\n(Spearman r={r1:.3f}, expect negative)")

# Delta-alpha vs expression divergence
axes[1].scatter(delta_alpha, expr_div, s=1, alpha=0.2, c="coral", rasterized=True)
r2 = spearmanr(delta_alpha, expr_div).correlation
axes[1].set_xlabel("Delta-alpha"); axes[1].set_ylabel("|log expr divergence|")
axes[1].set_title(f"Delta-Alpha vs Expression Divergence\n(Spearman r={r2:.3f})")

# Prior vs expression divergence (baseline)
axes[2].scatter(prior, expr_div, s=1, alpha=0.2, c="gray", rasterized=True)
r3 = spearmanr(prior, expr_div).correlation
axes[2].set_xlabel("Conservation prior"); axes[2].set_ylabel("|log expr divergence|")
axes[2].set_title(f"Prior vs Expression Divergence\n(Spearman r={r3:.3f})")

fig.tight_layout()
plt.show()

# Key test: alpha should correlate MORE strongly with expression divergence than prior alone
print(f"Alpha vs expr divergence:  r = {r1:.3f}")
print(f"Prior vs expr divergence:  r = {r3:.3f}")
print(f"Improvement: {abs(r1) - abs(r3):.3f} ({'alpha captures more' if abs(r1) > abs(r3) else 'no improvement'})")

## 5. S_g & FQR Analysis

S_g measures how well the shared decoder fits each gene beyond the null model. Positive S_g drives alpha ABOVE the prior (FM4 fix).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# S_g distribution
axes[0].hist(S_g, bins=80, color="teal", alpha=0.7, edgecolor="none")
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_xlabel("S_g (EMA)")
axes[0].set_title(f"S_g Distribution\n{(S_g>0).mean():.1%} positive, {(S_g==0).mean():.1%} exactly zero")

# S_g vs delta_alpha — the key FQR relationship
mask_pos = S_g > 0
axes[1].scatter(S_g[mask_pos], delta_alpha[mask_pos], s=1, alpha=0.3, c="teal", rasterized=True)
r_sg_da = spearmanr(S_g[mask_pos], delta_alpha[mask_pos]).correlation if mask_pos.sum() > 10 else 0
axes[1].set_xlabel("S_g (positive genes only)")
axes[1].set_ylabel("Delta-alpha")
axes[1].axhline(0, color="red", linestyle="--", alpha=0.3)
axes[1].set_title(f"S_g vs Delta-Alpha (r={r_sg_da:.3f})\nExpect positive: FQR pushes alpha up for well-fit genes")

# Correction vs S_g
axes[2].scatter(S_g, correction, s=1, alpha=0.3, c="purple", rasterized=True)
r_sg_c = spearmanr(S_g[mask_pos], correction[mask_pos]).correlation if mask_pos.sum() > 10 else 0
axes[2].set_xlabel("S_g"); axes[2].set_ylabel("GIN correction")
axes[2].axhline(0, color="red", linestyle="--", alpha=0.3)
axes[2].set_title(f"S_g vs Correction (r={r_sg_c:.3f})")

fig.tight_layout()
plt.show()

# Quantitative FQR effectiveness
print("=== FQR Effectiveness ===")
print(f"Genes with S_g > 0: {mask_pos.sum()} ({mask_pos.mean():.1%})")
print(f"Among S_g > 0 genes:")
print(f"  Mean delta_alpha: {delta_alpha[mask_pos].mean():.4f} (expect > 0)")
print(f"  Fraction delta_alpha > 0: {(delta_alpha[mask_pos] > 0).mean():.1%}")
print(f"Among S_g = 0 genes:")
mask_zero = S_g == 0
print(f"  Mean delta_alpha: {delta_alpha[mask_zero].mean():.4f}")
print(f"  Fraction delta_alpha > 0: {(delta_alpha[mask_zero] > 0).mean():.1%}")

# Mann-Whitney test
if mask_pos.sum() > 10 and mask_zero.sum() > 10:
    u, p = mannwhitneyu(delta_alpha[mask_pos], delta_alpha[mask_zero], alternative="greater")
    print(f"Mann-Whitney U test (S_g>0 has higher delta_alpha): p={p:.2e}")

## 6. Cross-Seed Alpha Reproducibility

Train with a different seed and compare alpha rankings. Stable rankings = robust biological signal.

In [ ]:
# Train a second model with different seed for reproducibility comparison
import time

torch.manual_seed(43)
np.random.seed(43)

adata2 = ad.read_h5ad(ADATA_PATH)
ChimeraModel.setup_anndata(adata2, species_key="species", batch_key="batch")
model2 = ChimeraModel.from_config(adata2, config_name="D",
                                   lambda_fit=0.1, lambda_anchor=0.001, lambda_res=0.01)

t0 = time.time()
model2.train(
    max_epochs=100, batch_size=256, early_stopping=False,
    plan_kwargs=dict(adv_weight=100, adv_lr=1e-4),
    accelerator="mps",
)
print(f"Seed 43 training: {time.time()-t0:.0f}s")

alpha2 = model2.get_alpha()
delta_alpha2 = model2.get_delta_alpha()

# Compare
r_alpha = spearmanr(alpha, alpha2).correlation
r_delta = spearmanr(delta_alpha, delta_alpha2).correlation

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(alpha, alpha2, s=1, alpha=0.2, c="steelblue", rasterized=True)
axes[0].plot([0, 1], [0, 1], "r--", linewidth=0.8)
axes[0].set_xlabel("Alpha (seed 42)"); axes[0].set_ylabel("Alpha (seed 43)")
axes[0].set_title(f"Alpha Reproducibility (Spearman r={r_alpha:.3f})")

axes[1].scatter(delta_alpha, delta_alpha2, s=1, alpha=0.2, c="coral", rasterized=True)
lim = max(abs(delta_alpha).max(), abs(delta_alpha2).max()) * 1.1
axes[1].plot([-lim, lim], [-lim, lim], "r--", linewidth=0.8)
axes[1].set_xlabel("Delta-alpha (seed 42)"); axes[1].set_ylabel("Delta-alpha (seed 43)")
axes[1].set_title(f"Delta-Alpha Reproducibility (Spearman r={r_delta:.3f})")

fig.tight_layout()
plt.show()

print(f"\n=== Reproducibility ===")
print(f"Alpha ranking:       Spearman r = {r_alpha:.3f} (target > 0.8)")
print(f"Delta-alpha ranking: Spearman r = {r_delta:.3f}")
print(f"{'PASS' if r_alpha > 0.8 else 'FAIL'}: Alpha ranking {'is' if r_alpha > 0.8 else 'is NOT'} reproducible across seeds")

## 7. Summary

Collect all biological validation results.

In [ ]:
summary = {
    "Alpha mean": f"{alpha.mean():.3f}",
    "Alpha std": f"{alpha.std():.3f}",
    "Delta-alpha range": f"[{delta_alpha.min():.3f}, {delta_alpha.max():.3f}]",
    "Delta-alpha positive (>0.02)": f"{(delta_alpha>0.02).mean():.1%}",
    "Delta-alpha negative (<-0.02)": f"{(delta_alpha<-0.02).mean():.1%}",
    "Alpha-prior Spearman r": f"{spearmanr(prior, alpha).correlation:.3f}",
    "Alpha-expr_div Spearman r": f"{r1:.3f}",
    "Prior-expr_div Spearman r": f"{r3:.3f}",
    "Alpha captures more expr info": f"{'YES' if abs(r1) > abs(r3) else 'NO'}",
    "FQR: S_g>0 fraction": f"{(S_g>0).mean():.1%}",
    "FQR: delta_alpha higher for S_g>0": f"p={p:.2e}" if 'p' in dir() else "N/A",
    "Cross-seed alpha Spearman r": f"{r_alpha:.3f}",
    "Cross-seed delta Spearman r": f"{r_delta:.3f}",
    "GIN correction mean |c|": f"{np.abs(correction).mean():.3f}",
}

print("=" * 55)
print("  BIOLOGICAL ANALYSIS SUMMARY")
print("=" * 55)
for k, v in summary.items():
    print(f"  {k:<38} {v}")
print("=" * 55)

# Overall assessment
issues = []
if abs(r1) <= abs(r3):
    issues.append("Alpha does not capture more expression info than prior")
if r_alpha < 0.8:
    issues.append(f"Alpha not reproducible across seeds (r={r_alpha:.3f})")
if (delta_alpha > 0.02).mean() < 0.05:
    issues.append("FM4: insufficient positive delta-alpha")

if issues:
    print("\nISSUES:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("\nAll biological validation checks PASSED.")